# Experiment 10 — Tissue × Relative Geometry Ordinal Tokens — Colab Run

MiT-B1 + CORN with tissue × relative-depth spatial tokens. Trains the new branch and the last RGB stage. Four matched runs: `rgb`, `tissue`, `geometry`, `full`; three folds each. The RGB reference is **81.68% accuracy**; no full exp10 result is available yet.

Uses the same Colab layout as exp3/5/7: clone the repo into `/content`, unzip data from Drive into `/content/data`, and save outputs to Drive. Select a **GPU runtime** before starting.

**Upload these three ZIP files to `MyDrive/thesis/` before running:**

| ZIP filename | Required contents after extraction |
|---|---|
| `wound-stage-classification.zip` | `wound-stage-classification/stage_1/` … `stage_4/`, containing the 1,091 original RGB images |
| `wound-staging-masks-exp-3.zip` | `wound-staging-masks-exp-3/stage_1/` … `stage_4/`, containing the 1,091 eight-channel `.npy` files from exp3 |
| `wound-stage-depth-anything-v2-small.zip` | `wound-stage-depth-anything-v2-small/stage_1/` … `stage_4/`, containing the 1,091 raw float32 `.npy` maps, **plus `manifest.json` and `provenance.json` at the dataset root** |

The ZIPs are prepared locally under the project's `datasets/` folder. Upload the ZIP files themselves; the notebook extracts them. Each ZIP must contain its named dataset folder, rather than loose `stage_*` folders.

**Also keep the three existing RGB baseline checkpoint files on Drive:**

```text
MyDrive/thesis/
├── wound-stage-classification.zip
├── wound-staging-masks-exp-3.zip
├── wound-stage-depth-anything-v2-small.zip
└── task_3_outputs/
    └── checkpoints/
        ├── mit_b1__corn_fold0.pt
        ├── mit_b1__corn_fold1.pt
        └── mit_b1__corn_fold2.pt
```

The `.pt` files do not need to be zipped. These are the original Task 3 MiT-B1+CORN checkpoints; this notebook checks them against the recorded baseline hashes. It does not retrain a missing baseline.

Outputs go to `MyDrive/thesis/exp10_tissue_geometry_tokens_outputs/`. The current exp10 code must be available in the GitHub repo cloned in Step 2. The full project folder is not required on Drive. Depth is read from the ZIP; no depth model or segmentation checkpoint needs to be downloaded here.

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Clone / update the repo

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_DIR = "/content/CanRollasThesisProject"
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "https://github.com/canrollas/CanRollasThesisProject.git", REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)

EXP_DIR = f"{REPO_DIR}/more_experiments/exp/exp10_tissue_geometry_tokens"
assert Path(EXP_DIR, "scripts/train.py").is_file(), "The cloned repo does not contain exp10 yet; update the GitHub repo first."
print("repo:", REPO_DIR)
print("experiment:", EXP_DIR)

## 3. Fetch the original PIID staging dataset

Reads `/content/drive/MyDrive/thesis/wound-stage-classification.zip`, the same original-RGB dataset used by task_3 and exp7. Extracts under `/content/data` and links into the repo's `datasets/` directory. The helper is reused for the next two ZIPs. Hidden macOS files are excluded so they cannot change fold membership.

In [ ]:
import zipfile

DRIVE_BASE = Path("/content/drive/MyDrive/thesis")
EXTRACT_DIR = Path("/content/data")

def fetch_dataset(dataset_name):
    zip_path = DRIVE_BASE / f"{dataset_name}.zip"
    assert zip_path.is_file(), f"Upload {zip_path.name} to MyDrive/thesis/ first."
    EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
    stages = [f"stage_{i}" for i in range(1, 5)]

    def find_roots():
        return sorted(p for p in EXTRACT_DIR.rglob(dataset_name)
                      if p.is_dir() and all((p / stage).is_dir() for stage in stages)
                      and not any(part.startswith('.') or part == '__MACOSX'
                                  for part in p.relative_to(EXTRACT_DIR).parts))

    if not find_roots():
        with zipfile.ZipFile(zip_path) as archive:
            members = []
            for member in archive.infolist():
                parts = Path(member.filename).parts
                if any(part.startswith('.') or part == '__MACOSX' for part in parts):
                    continue
                assert not Path(member.filename).is_absolute() and '..' not in parts, "Invalid ZIP path"
                members.append(member)
            archive.extractall(EXTRACT_DIR, members=members)

    candidates = find_roots()
    assert len(candidates) == 1, f"Expected one extracted '{dataset_name}' folder; found {candidates}"
    source = candidates[0].resolve()
    # Remove resource files left by an earlier manual unzip in this runtime.
    for stage in stages:
        for extra in (source / stage).iterdir():
            if extra.is_file() and extra.name.startswith('.'):
                extra.unlink()
    destination = Path(REPO_DIR) / "datasets" / dataset_name
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.is_symlink():
        destination.unlink()
    assert not destination.exists(), f"A real dataset directory already exists: {destination}"
    destination.symlink_to(source, target_is_directory=True)
    counts = {stage: sum(p.is_file() for p in (source / stage).iterdir()) for stage in stages}
    assert sum(counts.values()) == 1091, f"Expected 1,091 samples: {counts}"
    print("ZIP:", zip_path)
    print("dataset:", destination, "->", source)
    print("samples:", counts)
    return destination

DATASET_DIR_STAGE = fetch_dataset("wound-stage-classification")

## 4. Fetch the tissue-probability dataset

Reads `/content/drive/MyDrive/thesis/wound-staging-masks-exp-3.zip`, exactly the exp3/4/5 tissue dataset. Arrays contain RGB + five soft tissue channels; exp10 checks that cached RGB matches the original images. No Task 1/2 segmentation checkpoints are needed.

In [ ]:
DATASET_DIR_TISSUE = fetch_dataset("wound-staging-masks-exp-3")

## 5. Fetch the relative-depth dataset

Reads `/content/drive/MyDrive/thesis/wound-stage-depth-anything-v2-small.zip`. Include the full generated folder with `manifest.json`, `provenance.json` and all four `stage_*` directories. These are raw float32 relative inverse-depth maps, not PNG/JPEG visualizations or physical wound-depth measurements.

In [ ]:
DATASET_DIR_DEPTH = fetch_dataset("wound-stage-depth-anything-v2-small")
for filename in ("manifest.json", "provenance.json"):
    assert (DATASET_DIR_DEPTH / filename).is_file(), f"Missing {filename} in the depth ZIP"

## 6. Install dependencies

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{EXP_DIR}/requirements.txt"], check=True)
os.chdir(f"{EXP_DIR}/scripts")
print(os.getcwd())

## 7. RGB baseline checkpoints — fold0/1/2

Copies the original `mit_b1__corn_fold{0,1,2}.pt` files from `MyDrive/thesis/task_3_outputs/checkpoints/` into the same repo location used by exp5/7. Missing or different baseline files stop here. Exp10 pins the historical checkpoints, so they are not silently replaced by newly trained weights.

In [ ]:
import hashlib
import json
import shutil

CKPT_DIR = Path(REPO_DIR) / "task_3/results/task_3_outputs/checkpoints"
DRIVE_CKPT_DIR = DRIVE_BASE / "task_3_outputs/checkpoints"
CKPT_DIR.mkdir(parents=True, exist_ok=True)
protocol = json.loads(Path(EXP_DIR, "configs/baseline_protocol.json").read_text())

def file_sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

for fold in range(3):
    name = f"mit_b1__corn_fold{fold}.pt"
    source = DRIVE_CKPT_DIR / name
    destination = CKPT_DIR / name
    expected = protocol["folds"][str(fold)]["baseline_sha256"]
    if destination.is_file():
        assert file_sha256(destination) == expected, f"Local baseline differs: {destination}"
    else:
        assert source.is_file(), f"Upload the original {name} to {DRIVE_CKPT_DIR}"
        assert file_sha256(source) == expected, f"Drive baseline differs from the historical checkpoint: {source}"
        shutil.copy2(source, destination)
    print(f"fold{fold}: original RGB checkpoint verified")

## 8. Architecture and input sanity checks

Checks the model, augmentation alignment and exact RGB baseline path. Verifies all relative-depth source/output hashes after ZIP extraction, then verifies the original three-fold split. This does not start training.

In [ ]:
import torch
import yaml
assert torch.cuda.is_available(), "Select Runtime > Change runtime type > GPU in Colab."
print("GPU:", torch.cuda.get_device_name(0))

def run_and_print(command):
    result = subprocess.run(command, capture_output=True, text=True)
    print(result.stdout)
    print(result.stderr)
    result.check_returncode()
    return result

run_and_print([sys.executable, "-m", "unittest", "discover", "-s", ".", "-p", "test_*.py", "-v"])
run_and_print([sys.executable, "prepare_depth.py", "--verify-only"])

sys.path.insert(0, f"{EXP_DIR}/scripts")
from train import resolve_config, build_folds, validate_baseline_protocol
cfg = resolve_config(Path(EXP_DIR) / "configs/base.yaml")
folds = build_folds(cfg["data_root"], cfg["num_folds"], cfg["val_fraction"], cfg["split_seed"])
validate_baseline_protocol(cfg, folds)
print("All three baseline splits and checkpoints verified")

## 9. Redirect checkpoints/results to Drive

Best checkpoints, resume checkpoints, history and evaluation files are saved under `MyDrive/thesis/exp10_tissue_geometry_tokens_outputs/main/seed42/<mode>/`. A disconnected runtime does not lose these files. Data and repo execution stay under `/content`, matching the other experiments.

In [ ]:
OUTPUT_DIR = "/content/drive/MyDrive/thesis/exp10_tissue_geometry_tokens_outputs"
RUN_ROOT = Path(OUTPUT_DIR) / "main" / f"seed{cfg['train_seed']}"
os.makedirs(OUTPUT_DIR, exist_ok=True)

MODES = ("rgb", "tissue", "geometry", "full")
FOLDS = (0, 1, 2)
print("outputs:", OUTPUT_DIR)
print("runs:", RUN_ROOT)
print("modes:", MODES)
print("folds:", FOLDS)
print("epochs:", cfg["epochs"], "patience:", cfg["patience"], "batch size:", cfg["batch_size"])

## 10. Train — all three folds

Runs the matched RGB fine-tuning control, tissue control, wound-geometry control and full model. Uses `configs/base.yaml` unchanged: batch 16, up to 50 epochs, patience 15. Each fold loads its matching original RGB checkpoint. Training selects on validation; final test evaluation is in Step 13.

If the runtime disconnects, rerun the setup cells and this cell. `--resume` is added only for folds with a saved resume checkpoint; new folds start normally. Already completed folds return without adding epochs. Errors stop the cell so a mismatched input or checkpoint is not retried indefinitely.

In [ ]:
def run_streaming(command):
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                text=True, bufsize=1)
    for line in process.stdout:
        print(line, end="", flush=True)
    process.wait()
    if process.returncode:
        raise subprocess.CalledProcessError(process.returncode, command)

for mode in MODES:
    for fold in FOLDS:
        run_dir = RUN_ROOT / mode
        command = [sys.executable, "-u", "train.py", "--mode", mode, "--folds", str(fold),
                   "--workers", "4", "--device", "cuda", "--output-dir", OUTPUT_DIR]
        if (run_dir / f"fold{fold}.resume.pt").is_file():
            command.append("--resume")
        print(f"\n=== {mode} / fold{fold} ===", flush=True)
        run_streaming(command)
print("training finished")

## 11. Check training results

Twelve rows are expected after all four modes finish (three folds per mode). This table shows **validation** selection, not test accuracy.

In [ ]:
import pandas as pd

rows = []
for mode in MODES:
    for fold in FOLDS:
        summary_path = RUN_ROOT / mode / f"fold{fold}.summary.json"
        assert summary_path.is_file(), f"Training is incomplete: {summary_path}"
        rows.append({"mode": mode, **json.loads(summary_path.read_text())})
training_results = pd.DataFrame(rows)
display(training_results)
display(training_results.groupby("mode")["best_val_f1"].agg(["mean", "std"]))

## 12. Validation evaluation and prior ablations

Evaluates the selected checkpoints on validation. The full model includes real/zero/uniform/shuffled tissue and flat/shuffled depth conditions. Existing complete evaluations from the same selected checkpoints are reused when rerunning this cell. Four separately trained controls are compared together.

In [ ]:
from train import code_fingerprint, input_fingerprint
from compare_runs import compare_runs, write_comparison

def evaluate_split(split):
    all_samples = [sample for fold in folds for sample in fold["test"]]
    for mode in MODES:
        run_dir = RUN_ROOT / mode
        evaluation_dir = run_dir / f"evaluation_{split}"
        if evaluation_dir.exists():
            assert (evaluation_dir / "metrics.json").is_file() and (evaluation_dir / "predictions.csv").is_file(), f"Incomplete evaluation: {evaluation_dir}"
            report = json.loads((evaluation_dir / "metrics.json").read_text())
            assert report["mode"] == mode and report["split"] == split and report["complete_three_fold"] and not report["smoke"]
            assert report["code_fingerprint"] == code_fingerprint(), f"Evaluation code changed: {evaluation_dir}"
            assert report["input_fingerprint"] == input_fingerprint(cfg, all_samples, mode), f"Evaluation inputs changed: {evaluation_dir}"
            for fold in FOLDS:
                assert report["folds"][str(fold)]["checkpoint_sha256"] == file_sha256(run_dir / f"fold{fold}.pt"), f"Selected checkpoint changed; use a new output folder: {evaluation_dir}"
            print("reusing:", evaluation_dir)
        else:
            subprocess.run([sys.executable, "-u", "evaluate.py", "--run-dir", str(run_dir),
                            "--split", split, "--device", "cuda", "--workers", "4"], check=True)
    comparison_dir = RUN_ROOT / f"comparison_{split}"
    current_comparison = compare_runs(RUN_ROOT, split)
    if comparison_dir.exists():
        assert json.loads((comparison_dir / "metrics.json").read_text()) == current_comparison, f"Saved comparison is stale; use a new output folder: {comparison_dir}"
    else:
        write_comparison(current_comparison, comparison_dir)
    assert (comparison_dir / "summary.csv").is_file() and (comparison_dir / "metrics.json").is_file(), f"Incomplete comparison: {comparison_dir}"
    display(pd.read_csv(comparison_dir / "summary.csv"))
    return comparison_dir

VALIDATION_COMPARISON = evaluate_split("val")

## 13. Final test evaluation and comparison with 81.68%

Run this cell after fixing the architecture/training choices using validation. Uses the three held-out test folds and the same CORN decoder for every model. Reports historical RGB, remeasured original RGB and the separately fine-tuned RGB control. Test results are not used to change hyperparameters. A three-fold single-seed gain still needs independent confirmation.

In [ ]:
TEST_COMPARISON = evaluate_split("test")
comparison = json.loads((TEST_COMPARISON / "metrics.json").read_text())
print(json.dumps(comparison["target_check"], indent=2))
print("RGB baseline for reference: accuracy=0.8168+-0.0150 macro_f1=0.8182+-0.0138 qwk=0.9138+-0.0164")

## 14. Inspect the full model's tissue/depth contribution

Reads the saved test ablations; no additional training. Changes in logits or predictions under a shuffled prior are diagnostics, not proof of improved accuracy. All reports and per-image predictions remain under `OUTPUT_DIR`.

In [ ]:
report = json.loads((RUN_ROOT / "full/evaluation_test/metrics.json").read_text())
conditions = ("real", "zero_tissue", "uniform_tissue", "shuffle_tissue", "flat_depth", "shuffle_depth", "shuffle_both")
rows = []
for fold, fold_report in report["folds"].items():
    for condition in conditions:
        stats = fold_report[condition]
        rows.append({"fold": int(fold), "condition": condition,
                     "accuracy": stats["accuracy"], "macro_f1": stats["macro_f1"], "qwk": stats["qwk"],
                     "prediction_flips_vs_real": stats.get("versus_real", {}).get("prediction_flips", 0),
                     "mean_logit_l2_change": stats.get("mean_logit_l2_change", 0.0)})
display(pd.DataFrame(rows))
print("Saved outputs:", OUTPUT_DIR)